In [3]:
%pwd

'd:\\COURSES\\Math-AI\\final_project\\Mathematics-for-Artificial-Intelligence\\notebooks'

In [ ]:
import pandas as pd
import re

data_path = r"../data/spam.csv"
dataset = pd.read_csv(data_path, encoding='latin-1')

In [13]:
dataset = dataset[['v1', 'v2']]
dataset = dataset.rename(columns={'v1': 'label', 'v2': 'text'})

In [14]:
dataset.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [26]:
def encode_label(label):
    return 1 if label.lower() == 'spam' else 0

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '<NUM>', text)
    text = re.sub(r'[^\w\s<>]', '', text)
    return text

def tokenize(text):
    return text.split()

STOP_WORDS = {"i", "me", "my", "we", "our", "you", "your", "he", "him", "his", 
              "she", "her", "it", "its", "they", "them", "their", "what", "which", 
              "who", "this", "that", "am", "is", "are", "was", "were", "be", "been", 
              "have", "has", "had", "do", "does", "did", "a", "an", "the", "and", 
              "but", "if", "or", "because", "as", "until", "while", "of", "at", 
              "by", "for", "with", "about", "to", "from", "in", "out", "on", "off"}

def remove_stopwords(tokens):
    return [word for word in tokens if word not in STOP_WORDS]

def naive_stemmer(tokens):
    stemmed_tokens = []
    for word in tokens:
        if word == "<NUM>":
            stemmed_tokens.append(word)
            continue
            
        if word.endswith('ing') and len(word) > 4: 
            word = word[:-3]
        elif word.endswith('ly') and len(word) > 3: 
            word = word[:-2]
        elif word.endswith('es') and len(word) > 4:
            word = word[:-2]
        elif word.endswith('s') and len(word) > 3 and not word.endswith('ss'): 
            word = word[:-1]
        stemmed_tokens.append(word)
    return stemmed_tokens

In [24]:
def text_preprocessing_pipeline(text):
    text = str(text)
    cleaned_text = clean_text(text)
    tokens = tokenize(cleaned_text)
    filltered_tokens = remove_stopwords(tokens)
    stemmed_tokens = naive_stemmer(filltered_tokens)

    return " ".join(stemmed_tokens)

In [27]:
dataset['processed_text'] = dataset['text'].apply(text_preprocessing_pipeline)
dataset.head()

,label,text,processed_text
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available on bugi n grea...
1,ham,Ok lar... Joking wif u oni...,ok lar jok wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry <NUM> wk comp win fa cup final tkt ...
3,ham,U dun say so early hor... U c already then say...,u dun say so ear hor u c already then say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah dont think goe usf liv around here though


In [28]:
dataset['encoded_label'] = dataset['label'].apply(encode_label)
dataset.head()

,label,text,processed_text,encoded_label
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available on bugi n grea...,0
1,ham,Ok lar... Joking wif u oni...,ok lar jok wif u oni,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry <NUM> wk comp win fa cup final tkt ...,1
3,ham,U dun say so early hor... U c already then say...,u dun say so ear hor u c already then say,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah dont think goe usf liv around here though,0


In [35]:
dataset.to_csv("processed_spam.csv", index=False, encoding="utf-8")